In [ ]:
from datetime import datetime, timedelta
import mysql.connector
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import shap as sp
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm

In [ ]:
import sqlalchemy
from utils import load_sql_query, Engine, logger
from urllib.parse import quote_plus
from dotenv import load_dotenv
import os

# Force pymysql instead of mysqlconnector
load_dotenv()
username = os.getenv("DATABASE_USERNAME")
password = quote_plus(os.getenv("DATABASE_PASSWORD"))
host = os.getenv("DATABASE_HOST")
database = os.getenv("DATABASE_NAME")

engine = Engine()  # create normally
engine.engine = sqlalchemy.create_engine(  # then swap the internal engine
    f'mysql+pymysql://{username}:{password}@{host}/{database}',
    pool_recycle=300,
    pool_pre_ping=True,
)

query = load_sql_query("query.sql")
dataframe = pd.read_sql(query, engine.engine)
dataframe.head()

In [ ]:
# Función de probabilidad acumulada por día, con corrección para evitar descenso
# 1. Aplicar valores datetime a las columnas relevantes
dataframe['created'] = pd.to_datetime(dataframe['created'], errors='coerce')
dataframe['event_date'] = pd.to_datetime(dataframe['event_date'], errors='coerce')

# 2. Calcular duración en días entre created y event_date
duration = (dataframe['event_date'] - dataframe['created']).dt.days + 1

# Mantener filas válidas
valid_mask = duration.ge(1) & dataframe['created'].notna() & dataframe['event_date'].notna()
invalid_rows = (~valid_mask).sum()
if invalid_rows > 0:
    print(f"Rows excluded due to invalid/negative duration: {invalid_rows}")

dataframe_valid = dataframe.loc[valid_mask].copy()
dataframe_valid['duration'] = duration.loc[valid_mask].astype(int)

# 3. Determinar el horizonte máximo de días (ej. 10 días en tu imagen)
max_days = dataframe_valid['days_to_event'].max()

# 4. Expandir CADA propiedad por el max_days (no solo por su propia duración)
dataframe_expanded = dataframe_valid.loc[dataframe_valid.index.repeat(max_days)].copy()

# 5. Crear el contador de días global para cada unidad
dataframe_expanded['day_of_observation'] = dataframe_expanded.groupby(level=0).cumcount() + 1

# 6. Calcular la probabilidad con un tope (CLIP) en 1.0
# Esto asegura que si una unidad terminaba en el día 5, del día 6 al 10 valga 1.
dataframe_expanded['prob_day'] = (
    dataframe_expanded['day_of_observation'] / dataframe_expanded['days_to_event']
).clip(upper=1.0)

# 7. Ahora el promedio por día será constante o ascendente, nunca descendente
average_per_day = dataframe_expanded.groupby('day_of_observation')['prob_day'].mean()

# 8. Separar bases de churn y arrendados para graficar ambos en el mismo gráfico
average_per_day_churn = dataframe_expanded[dataframe_expanded['churn'] == 1]
average_per_day_arrendado = dataframe_expanded[dataframe_expanded['ha_sido_arrendada'] == 'Si']

In [ ]:
# 2. Graficar curvas de CDFs
x = np.arange(max(average_per_day.index.max(), 30))  # Show at least 30 days on x-axis
width = 0.25

# Wider figure for more horizontal spacing between days
fig, ax1 = plt.subplots(figsize=(max(16, len(x) * 0.25), 7))

# X axis
ax1.plot(x - 0.25, average_per_day_churn, color='#4C84A8', alpha=0.85, label='1 unidad')
ax1.set_xlabel('Día de Observación')
ax1.set_ylabel('Probabilidad Acumulada')

# 5. X-ticks: show every day, rotated 90° to avoid overlap
ax1.set_xticks(x)
ax1.set_xticklabels(x, rotation=90, fontsize=8)
ax1.tick_params(axis='x', which='major', pad=4)

# 6. Combined legend
h1, l1 = ax1.get_legend_handles_labels()
ax1.legend(h1, l1, title='Estado', loc='upper left')

# 7. Lines every 7 days for better readability
for day in range(0, len(x), 7):
    ax1.axvline(day, color='black', linestyle=':', linewidth=0.8, alpha=0.5) 

ax1.axhline(y=0.25, color='orange', linestyle='--', linewidth=0.8, alpha=0.5, label='Referencia 25%')
ax1.axhline(y=0.5, color='red', linestyle='--', linewidth=0.8, alpha=0.5, label='Referencia 50%')
ax1.axhline(y=0.75, color='orange', linestyle='--', linewidth=0.8, alpha=0.5, label='Referencia 75%')

plt.legend()
plt.title('Promedio Probabilidad Acumulada por Día por Cantidad de unidades del owner')
plt.tight_layout()
plt.show()